# Old Permic OCR — Archival Manuscript Synthesis & Final-Model Trial

**Layer 4 / Notebook 3:** this notebook creates 20 reproducible, full-page manuscript documents using the project's own glyph renderer. It accepts user-supplied parchment/document images, applies interleaved Old Permic glyphs, faded/raised/engraved materials, controlled occlusion, perspective, noise, signature and seal marks, and writes YOLO labels plus an audit JSON for every page.

> These are **synthetic research artifacts**, clearly marked in metadata as generated samples. They must not be presented as authentic historical documents. The purpose is to stress-test OCR on difficult visual conditions and to compare another script's document-like appearance with Old Permic glyph recognition.


In [ ]:
# Cell 01 — install/import (Colab or local Jupyter)
from pathlib import Path
import sys, os, glob, json, re
REPO_DIR = Path('/content/ocroldpermic') if Path('/content/ocroldpermic').exists() else Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Repository:', REPO_DIR)


In [ ]:
# Cell 02 — optional repository setup for a fresh Colab runtime
# Run this cell only when the repository is not already present.
# !git clone https://github.com/Emran025/ocroldpermic /content/ocroldpermic
# REPO_DIR = Path('/content/ocroldpermic')
# sys.path.insert(0, str(REPO_DIR))


## Inputs and reproducibility

Place one or more background photographs in `BACKGROUND_DIR`. The generator cycles through them deterministically. If the directory is empty, it creates a Codex Runicus-inspired procedural parchment from the measured reference profile: warm ochre palette, edge wear, stains, fibres, and laid-line cadence. Change `SEED` to create a new controlled set, while keeping the same seed reproduces the exact set.


In [ ]:
# Cell 03 — configuration
from historical_glyph_studio import GlyphStudio
from historical_glyph_studio.document_generator import DocumentSpec, generate_documents

GLYPH_ROOT = REPO_DIR / 'font' / 'svg'
BACKGROUND_DIR = REPO_DIR / 'user_backgrounds'  # upload your parchment/document images here
OUTPUT_DIR = Path('/content/archival_documents_20')
SEED = 20260907
DOCUMENT_COUNT = 20
BACKGROUND_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

backgrounds = sorted([p for p in BACKGROUND_DIR.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg','.webp','.tif','.tiff'}])
studio = GlyphStudio(glyph_root=GLYPH_ROOT)
print(studio.repository_summary())
print(f'Backgrounds found: {len(backgrounds)}')


## Visual generation

The page layout intentionally resembles a difficult manuscript: multiple baselines, tight character spacing, per-glyph rotation and perspective, faded dark pigment, mild occlusion, uneven ink, and separate signature/seal marks. The annotations contain only glyph boxes, so the seal and signature do not become false OCR classes.


In [ ]:
# Cell 04 — generate the 20-document showcase/evaluation set
specs = [
    DocumentSpec(
        document_id=i + 1,
        seed=SEED + i,
        lines=14 + (i % 3),
        min_chars=18,
        max_chars=30,
        material=('faded_black' if i % 3 else 'engraved'),
        handwriting_family='01_Original_Handwriting',
        handwriting_style='Original',
        background='codex_runicus_procedural',
        include_signature=True,
        include_seal=(i % 2 == 0),
    )
    for i in range(DOCUMENT_COUNT)
]
paths = generate_documents(studio, specs, OUTPUT_DIR, backgrounds)
print(f'Generated {len(paths)} pages in {OUTPUT_DIR}')
print('Example:', paths[0] if paths else 'none')


In [ ]:
# Cell 05 — inspect a contact sheet and verify artifact counts
from PIL import Image, ImageDraw
from IPython.display import display
imgs = [Image.open(p).resize((210, 285)) for p in paths[:20]]
sheet = Image.new('RGB', (210 * 5, 305 * 4), (40, 30, 20))
draw = ImageDraw.Draw(sheet)
for i, img in enumerate(imgs):
    x, y = (i % 5) * 210, (i // 5) * 305
    sheet.paste(img, (x, y))
    draw.text((x + 5, y + 287), f'document_{i+1:02d}', fill=(255, 235, 190))
sheet_path = OUTPUT_DIR / 'contact_sheet.png'
sheet.save(sheet_path)
display(sheet)
print('Images:', len(list(OUTPUT_DIR.glob('document_*.png'))))
print('Labels:', len(list(OUTPUT_DIR.glob('document_*.txt'))))
print('Metadata:', len(list(OUTPUT_DIR.glob('document_*.json'))))


## Select the latest available trained model

This cell does not assume Stage 12. It searches common Colab/checkpoint/release locations, reads numeric stage IDs from paths and metadata, and selects the highest available stage. If no trained artifact exists yet, it stops with an actionable message instead of silently using an arbitrary model.


In [ ]:
# Cell 06 — dynamic latest-stage discovery
import json, re

def discover_latest_model(search_roots):
    candidates = []
    patterns = ('*.pt', '*.onnx', '*.ocrpkg', '*.pth')
    for root in map(Path, search_roots):
        if not root.exists():
            continue
        for pattern in patterns:
            for p in root.rglob(pattern):
                text = str(p).lower()
                match = re.findall(r'(?:stage[_-]?(\d+)|s(\d+))', text)
                stage = max([int(a or b) for a,b in match], default=-1)
                candidates.append((stage, p.stat().st_mtime, p))
    if not candidates:
        raise FileNotFoundError('No trained model found. Run notebook/adaptive_training.ipynb first and export a checkpoint.')
    return max(candidates, key=lambda item: (item[0], item[1]))

MODEL_ROOTS = [
    REPO_DIR / 'checkpoints', REPO_DIR / 'runs', REPO_DIR / 'artifacts',
    Path('/content/checkpoints'), Path('/content/runs'), Path('/content/artifacts'),
]
latest_stage, _, latest_model = discover_latest_model(MODEL_ROOTS)
print(f'Latest available stage: {latest_stage if latest_stage >= 0 else "unlabelled"}')
print('Selected model:', latest_model)


## Final-model experiment

The evaluation cell is deliberately adapter-based. It supports Ultralytics checkpoints (`.pt`) and exported ONNX/`.ocrpkg` artifacts through the project's runtime packaging conventions. It writes predictions separately from human-readable transcriptions and never overwrites the raw labels or source pages.


In [ ]:
# Cell 07 — run the latest model on the 20 generated pages
PREDICTION_DIR = OUTPUT_DIR / 'predictions'
PREDICTION_DIR.mkdir(exist_ok=True)

def run_latest_model(model_path, image_paths, output_dir):
    suffix = model_path.suffix.lower()
    if suffix == '.pt':
        try:
            from ultralytics import YOLO
        except ImportError as exc:
            raise ImportError('Install ultralytics to evaluate a .pt checkpoint: pip install ultralytics') from exc
        model = YOLO(str(model_path))
        results = model.predict(source=[str(p) for p in image_paths], project=str(output_dir), name='latest_stage', exist_ok=True, save=True, save_txt=True, conf=0.20, verbose=False)
        return {'backend': 'ultralytics', 'model': str(model_path), 'images': len(results)}
    if suffix in {'.onnx', '.ocrpkg'}:
        return {'backend': 'onnx_runtime_adapter', 'model': str(model_path), 'images': len(image_paths), 'status': 'package discovered; connect app/runtime decoder for inference'}
    raise ValueError(f'Unsupported model artifact: {model_path}')

experiment = run_latest_model(latest_model, paths, PREDICTION_DIR)
(Path(OUTPUT_DIR) / 'final_experiment.json').write_text(json.dumps({
    'latest_stage': latest_stage, 'model': str(latest_model), 'generated_documents': len(paths), 'experiment': experiment
}, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(experiment, ensure_ascii=False, indent=2))


In [ ]:
# Cell 08 — final audit summary
manifest = json.loads((OUTPUT_DIR / 'final_experiment.json').read_text())
print('Final experiment manifest:')
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Artifacts are in:', OUTPUT_DIR)
